# Assignment 9.2 — Deep Q-Learning (DQN)

Same `JumperEnv`, but the discretization scaffolding from Part 1 is
gone: a neural net consumes the raw 5-dim feature vector.

DQN (Mnih et al. 2013/2015) replaces the Q-table with a network
$Q_\theta(s, a)$ and adds two ingredients to make off-policy TD
learning stable under function approximation:

1. **Experience replay** — transitions $(s, a, r, s', \text{done})$ are
   stored in a ring buffer and sampled in i.i.d. mini-batches,
   breaking temporal correlation in the gradient.
2. **Target network** — a periodically-updated copy $Q_{\theta^-}$
   produces the bootstrap target

   $$
   y = r + \gamma \cdot (1 - \text{done}) \cdot \max_{a'} Q_{\theta^-}(s', a')
   $$

   so the moving target doesn't chase its own tail.

Loss for a batch $B$:

$$
\mathcal{L}(\theta) = \frac{1}{|B|} \sum_{(s,a,r,s',d)} \big( Q_\theta(s, a) - y \big)^2
$$

Action selection is again ε-greedy with decay.

## Tasks

Fill in the `TODO` blocks below:

1. **`QNetwork.__init__` / `forward`** — a small MLP:
   `5 → hidden → hidden → 2`, ReLU activations, no activation on the
   output.
2. **`ReplayBuffer.push` / `sample`** — store transitions, return
   five numpy arrays of shape `(batch_size, ...)`.
3. **`DQNAgent.act`** — ε-greedy, with `torch.no_grad()` around the
   forward pass.
4. **`DQNAgent.learn`** — sample a batch, compute the TD target with
   `self.target_net` (under `torch.no_grad`), Huber loss, Adam step,
   gradient clip.
5. **`DQNAgent.update_target`** — hard copy of the online network's
   parameters into the target network.

## Hints

- Defaults: hidden=128, batch=64, replay capacity=100k, γ=0.99,
  lr=5e-4, Huber loss, gradient clip at L2-norm 10, ε: 1.0 → 0.05
  linearly over the first 50k env steps, warm-up = 2000 transitions,
  target update every 1000 env steps. Train for 150k env steps
  (~3–4 min on a laptop CPU).
- Use **Huber loss** (`F.smooth_l1_loss`) — standard DQN choice,
  forgiving of outliers in the TD target.
- **Normalize observations** before feeding the net.
  `JumperEnv.FEATURE_LOW` / `FEATURE_HIGH` give the bounds.
- **Gradient clipping** matters here: TD-target spikes (e.g. when a
  collision lands on a previously-good Q estimate) can destabilise
  the net.
- **What "works" looks like.** A converged vanilla DQN on this env
  reliably beats the random baseline (~70 steps) by a wide margin
  and reaches an avg-100 episode return of ~10. Greedy evaluation
  across seeds is **high-variance**: some seeds may clear 50+
  obstacles, others die early. This is expected for vanilla DQN on
  a small problem — modern fixes (Double DQN, Dueling, prioritised
  replay) flatten that variance but are out of scope here.

In [ ]:
from __future__ import annotations
from collections import deque
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from arena import JumperEnv

## 1. Q-network

In [ ]:
class QNetwork(nn.Module):
    """
    A small MLP: obs_dim -> hidden -> hidden -> n_actions

    TODO:
      - in __init__, build self.net as a nn.Sequential of:
          Linear(obs_dim, hidden) -> ReLU -> Linear(hidden, hidden) -> ReLU -> Linear(hidden, n_actions)
      - in forward, return self.net(x)
    """

    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        # TODO
        raise NotImplementedError("QNetwork.__init__")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        raise NotImplementedError("QNetwork.forward")

## 2. Replay buffer

In [ ]:
class ReplayBuffer:
    """A simple FIFO buffer of (s, a, r, s', done) transitions."""

    def __init__(self, capacity: int = 100_000, seed: int | None = None):
        self.capacity = capacity
        self.buf: deque = deque(maxlen=capacity)
        self.rng = np.random.default_rng(seed)

    def __len__(self) -> int:
        return len(self.buf)

    def push(self, s, a, r, s_next, done) -> None:
        """
        TODO: append (s, a, r, s_next, done) to self.buf. Make sure s
        and s_next are stored as numpy float32 arrays so torch tensor
        conversion is cheap in sample().
        """
        raise NotImplementedError("ReplayBuffer.push")

    def sample(self, batch_size: int):
        """
        Return five numpy arrays:
            states     (batch, obs_dim) float32
            actions    (batch,)         int64
            rewards    (batch,)         float32
            next_states(batch, obs_dim) float32
            dones      (batch,)         float32   (1.0 if done else 0.0)

        TODO: draw `batch_size` random indices via self.rng.integers
        and stack.
        """
        raise NotImplementedError("ReplayBuffer.sample")

## 3. The agent

In [ ]:
@dataclass
class DQNAgent:
    obs_dim: int = 5
    n_actions: int = 2
    hidden: int = 128

    gamma: float = 0.99
    lr: float = 5e-4
    batch_size: int = 64
    buffer_capacity: int = 100_000
    warmup: int = 2_000
    target_update_every: int = 1_000
    grad_clip: float = 10.0

    epsilon_start: float = 1.0
    epsilon_end: float = 0.05
    epsilon_decay_steps: int = 50_000

    obs_low: np.ndarray = field(default=None)
    obs_high: np.ndarray = field(default=None)

    seed: int | None = None
    device: str = "cpu"

    online_net: QNetwork = field(init=False)
    target_net: QNetwork = field(init=False)
    optimizer: torch.optim.Optimizer = field(init=False)
    buffer: ReplayBuffer = field(init=False)
    steps: int = field(init=False, default=0)
    rng: np.random.Generator = field(init=False)

    def __post_init__(self) -> None:
        if self.seed is not None:
            torch.manual_seed(self.seed)
        self.rng = np.random.default_rng(self.seed)
        self.online_net = QNetwork(self.obs_dim, self.n_actions, self.hidden).to(self.device)
        self.target_net = QNetwork(self.obs_dim, self.n_actions, self.hidden).to(self.device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        for p in self.target_net.parameters():
            p.requires_grad_(False)
        self.optimizer = torch.optim.Adam(self.online_net.parameters(), lr=self.lr)
        self.buffer = ReplayBuffer(self.buffer_capacity, seed=self.seed)
        if self.obs_low is None:  self.obs_low  = np.zeros(self.obs_dim, dtype=np.float32)
        if self.obs_high is None: self.obs_high = np.ones(self.obs_dim, dtype=np.float32)

    @property
    def epsilon(self) -> float:
        frac = min(1.0, self.steps / self.epsilon_decay_steps)
        return self.epsilon_start + frac * (self.epsilon_end - self.epsilon_start)

    def _normalise(self, obs: np.ndarray) -> np.ndarray:
        return ((obs - self.obs_low) / (self.obs_high - self.obs_low + 1e-8)).astype(np.float32)

    def act(self, obs: np.ndarray, greedy: bool = False) -> int:
        """
        ε-greedy action selection.

        TODO:
          - if not greedy and self.rng.random() < self.epsilon: return random action
          - otherwise:
                x = torch.from_numpy(self._normalise(obs)).unsqueeze(0).to(self.device)
                with torch.no_grad():
                    q = self.online_net(x)
                return int(q.argmax(dim=1).item())
        """
        raise NotImplementedError("DQNAgent.act")

    def learn(self) -> float | None:
        """
        Run one gradient step on a sampled mini-batch.
        Returns loss as a float, or None if not enough samples yet.

        TODO:
          if len(self.buffer) < max(self.warmup, self.batch_size):
              return None
          s, a, r, s_next, done = self.buffer.sample(self.batch_size)
          # convert to tensors on self.device — observations are already
          # normalised (we normalise before pushing to the buffer)
          s_t  = torch.from_numpy(s).to(self.device)
          sn_t = torch.from_numpy(s_next).to(self.device)
          a_t  = torch.from_numpy(a).to(self.device)
          r_t  = torch.from_numpy(r).to(self.device)
          d_t  = torch.from_numpy(done).to(self.device)

          q = self.online_net(s_t).gather(1, a_t.unsqueeze(1)).squeeze(1)
          with torch.no_grad():
              q_next = self.target_net(sn_t).max(dim=1).values
              y = r_t + self.gamma * (1.0 - d_t) * q_next
          loss = F.smooth_l1_loss(q, y)

          self.optimizer.zero_grad()
          loss.backward()
          torch.nn.utils.clip_grad_norm_(self.online_net.parameters(), self.grad_clip)
          self.optimizer.step()
          return float(loss.item())
        """
        raise NotImplementedError("DQNAgent.learn")

    def update_target(self) -> None:
        """
        Hard copy of online_net parameters into target_net.

        TODO: self.target_net.load_state_dict(self.online_net.state_dict())
        """
        raise NotImplementedError("DQNAgent.update_target")

    # ---- bookkeeping called by the training loop ----------------

    def remember(self, s, a, r, s_next, done) -> None:
        self.buffer.push(self._normalise(s), a, r, self._normalise(s_next), done)

    def step_done(self) -> None:
        self.steps += 1
        if self.steps % self.target_update_every == 0:
            self.update_target()

## 4. Run training

~150k env steps takes 3–4 minutes on a laptop CPU. Reduce `total_steps`
below if you just want to verify your TODOs are wired correctly.

In [ ]:
env = JumperEnv(seed=0)
agent = DQNAgent(
    obs_low=JumperEnv.FEATURE_LOW,
    obs_high=JumperEnv.FEATURE_HIGH,
    seed=0,
)

total_steps = 150_000
returns: list[float] = []
ep_return = 0.0
ep = 0
obs, _ = env.reset()

while agent.steps < total_steps:
    a = agent.act(obs)
    next_obs, r, term, trunc, info = env.step(a)
    agent.remember(obs, a, r, next_obs, term)
    agent.learn()
    agent.step_done()

    obs = next_obs
    ep_return += r

    if term or trunc:
        returns.append(ep_return)
        ep += 1
        if ep % 100 == 0:
            avg = np.mean(returns[-100:])
            print(f"step {agent.steps:6d}  ep {ep:5d}  "
                  f"return(avg100) {avg:7.2f}  ε {agent.epsilon:.3f}  "
                  f"buf {len(agent.buffer)}")
        obs, _ = env.reset()
        ep_return = 0.0

print(f"\ntraining done — {ep} episodes, {agent.steps} env steps.")

## 5. Learning curve

In [ ]:
r = np.asarray(returns, dtype=np.float32)
window = 25
smoothed = np.convolve(r, np.ones(window) / window, mode="valid")
plt.figure(figsize=(8, 4))
plt.plot(r, alpha=0.25, label="episode return")
plt.plot(np.arange(len(smoothed)) + window - 1, smoothed,
         label=f"rolling mean (w={window})")
plt.xlabel("episode"); plt.ylabel("return")
plt.title("DQN on JumperEnv")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Greedy evaluation

In [ ]:
agent.online_net.eval()
steps_log, score_log = [], []
for trial in range(10):
    obs, _ = env.reset(seed=200 + trial)
    steps, ret = 0, 0.0
    while True:
        a = agent.act(obs, greedy=True)
        obs, r, term, trunc, info = env.step(a)
        ret += r; steps += 1
        if term or trunc:
            break
    steps_log.append(steps); score_log.append(info["score"])
    print(f"  trial {trial}: steps={steps:5d}  return={ret:7.2f}  score={info['score']}")

print(f"\nmean steps {np.mean(steps_log):.1f}   mean score {np.mean(score_log):.1f}   "
      f"score std {np.std(score_log):.1f}")

## 7. Watch a learned episode

Roll out a greedy episode and play it back as an embedded animation.
Vanilla DQN is high-variance across seeds (some seeds clear 50+
obstacles, others die immediately) so the cell tries the same
10 eval seeds and picks the longest-surviving one for playback.
The cap is 250 frames (~8 s at 30 fps) so the notebook stays small
on the lucky seeds.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

agent.online_net.eval()
best = {"frames": [], "info": {"score": 0}, "actions": [], "seed": -1}
for trial_seed in range(200, 210):
    play_env = JumperEnv(seed=trial_seed)
    obs, _ = play_env.reset()
    frames = [play_env.render(mode="rgb_array")]
    actions: list[int] = []
    last_info = {"score": 0}
    while len(frames) < 250:
        a = agent.act(obs, greedy=True)
        actions.append(a)
        obs, r, term, trunc, last_info = play_env.step(a)
        frames.append(play_env.render(mode="rgb_array"))
        if term or trunc:
            break
    play_env.close()
    if len(frames) > len(best["frames"]):
        best = {"frames": frames, "info": last_info,
                "actions": actions, "seed": trial_seed}

frames = best["frames"]
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.axis("off")
im = ax.imshow(frames[0])

def _update(i):
    im.set_array(frames[i])
    return [im]

anim = animation.FuncAnimation(
    fig, _update, frames=len(frames), interval=33, blit=True)
plt.close(fig)
print(f"best seed: {best['seed']}   frames {len(frames)}   "
      f"score {best['info']['score']}   "
      f"jumps {sum(best['actions'])}/{len(best['actions'])}")
HTML(anim.to_jshtml())

## 8. Reflection

Answer briefly:

1. With the same env, your DQN converges in roughly **how many env
   steps**? Compare to your Q-learning agent from Part 1 (in
   *episodes* and in *env steps* — they aren't the same!).
2. **Disable the target network** (`target_update_every = 1`, so it
   tracks the online net every step). What happens to the learning
   curve, and why?
3. **Disable experience replay** (sample only the most recent
   transition each update — set `batch_size = 1` and remove the
   random sampling, just use the last transition). What happens?

## Why DQN here when Q-learning works?

Because the state space is *small enough* that tabular Q-learning
works. The point of this notebook is to verify that DQN reproduces
a tabular-style result on a problem you already understand — and
to feel the new failure modes (instability without target net or
replay, sensitivity to TD-target outliers, high variance across
seeds) that don't exist in the tabular setting. On richer
observations (try `obs_mode="rgb"` and a small CNN if you want
extra credit), the table approach is no longer feasible and DQN
becomes essential.

## Save the trained network

In [ ]:
import torch
from pathlib import Path
out = Path("dqn.pt")
torch.save(agent.online_net.state_dict(), out)
print(f"saved {out}")